In [1]:
import pandas as pd
from pathlib import Path

Option 1: Real data file

In [2]:
def test_real_data_file():
    """Importing a real data file to a test"""

    # Path to example test data
    csv_path = Path(__file__).parent.joinpath("data/patient_data.csv")

    # Load the data and check it was read
    df = import_patient_data(csv_path)
    assert not df.empty

Option 2: Temporary file

In [3]:
def test_temporary_file(tmp_path):
    """Providing data to a test via a temporary file"""

    # Create sample patient data
    testdata = pd.DataFrame(
        [["p1", "2024-01-01", "08:00", "2024-01-01", "09:00"]],
        columns=[
            "PATIENT_ID", "ARRIVAL_DATE", "ARRIVAL_TIME",
            "SERVICE_DATE", "SERVICE_TIME",
        ],
    )

    # Create a temporary CSV file
    csv_path = tmp_path / "patients.csv"
    testdata.to_csv(csv_path, index=False)

    # Load the data and check it was read
    df = import_patient_data(csv_path)
    assert not df.empty

Option 3: Mocking

In [4]:
def test_mocking(monkeypatch):
    """Providing data to a test via mocking"""

    # Create sample patient data
    testdata = pd.DataFrame(
        [["p1", "2024-01-01", "08:00", "2024-01-01", "09:00"]],
        columns=[
            "PATIENT_ID", "ARRIVAL_DATE", "ARRIVAL_TIME",
            "SERVICE_DATE", "SERVICE_TIME",
        ],
    )

    # Define a fake CSV reader that just returns our DataFrame
    def mock_read_csv(path):
        return testdata

    # Temporarily replace pd.read_csv with our fake version
    monkeypatch.setattr(pd, "read_csv", mock_read_csv)

    # Call the function with any path - it does not matter - it will use the
    # mocked reader, and pd.read_csv is never actually called
    df = import_patient_data("does_not_matter.csv")
    assert not df.empty

Design for testability

In [5]:
def get_patient_data(path):
    """I/O layer: read data from disk."""
    return pd.read_csv(path)


def process_patient_data(df):
    """Logic layer: all processing goes here, no file I/O."""
    # do whatever transformation you need
    return df


def import_patient_data(path):
    """Public entry point."""
    df = get_patient_data(path)
    return process_patient_data(df)